In [8]:
from pathlib import Path

import pandas as pd

In [9]:
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next(
    (root / "data" for root in SEARCH_ROOTS if (root / "data" / "olist_orders_dataset.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("프로젝트 루트의 data 폴더를 찾을 수 없습니다.")

customers_df = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geolocation_df = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
items_df = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments_df = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews_df = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
orders_df = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products_df = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers_df = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_df = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

# 데이터 전처리

## 1. canceled 행 삭제

In [10]:
# 확인
orders_df[orders_df["order_status"] == "canceled"]

# 삭제
orders_df = orders_df[orders_df["order_status"] != "canceled"]

## 2. seller_city가 seller_state에 포함되지 않은 이슈 해결

In [11]:
# seller의 zip_code로 state 붙이기
geo_zip_state = (
    geolocation_df[["geolocation_zip_code_prefix", "geolocation_state"]]
    .drop_duplicates()
    .rename(columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix"
    })
)

sellers_check = sellers_df.merge(
    geo_zip_state,
    on="seller_zip_code_prefix",
    how="left"
)

# seller_state와 geolocation_state 비교
sellers_check[sellers_check["seller_state"] != sellers_check["geolocation_state"]]

# 결측치 확인
sellers_check[
    (sellers_check["seller_state"] != sellers_check["geolocation_state"]) |
    (sellers_check["geolocation_state"].isna())
]

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_state
70,f410c8873029fcc3809b9df6d0b28914,95076,caxias do sul,SP,RS
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN,RJ
198,1284de4ae8aa26997e748c851557cf0e,85301,laranjeiras do sul,SP,PR
206,8b181ee5518df84f18f4e1a43fe07923,87360,goioere,SP,PR
275,3d700782d7818f2c1e0d7a9e9d75fc00,86170,sertanopolis,SP,PR
311,f626e15b7314c267e4429010866f70e9,85960,marechal candido rondon,SP,PR
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP,RJ
424,c8771b1a10bb99bb34d3c459c5cffb53,36512,tocantins,SP,MG
473,5962468f885ea01a1b6a97a218797b0a,82040,curitiba,PR,NaN
513,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,RJ


## 3. 데이터 프레임 합치기

In [12]:
# 데이터 합치기
olist_df = pd.merge(orders_df, payments_df, on = 'order_id')
olist_df = olist_df.merge(customers_df, on = 'customer_id')
olist_df = olist_df.merge(items_df, on = 'order_id')
olist_df = olist_df.merge(products_df, on = 'product_id')
olist_df = olist_df.merge(category_df, on = 'product_category_name')
olist_df = olist_df.merge(reviews_df, on = 'order_id')
olist_df = olist_df.merge(sellers_df, on = 'seller_id')

olist_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115073 entries, 0 to 115072
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       115073 non-null  str    
 1   customer_id                    115073 non-null  str    
 2   order_status                   115073 non-null  str    
 3   order_purchase_timestamp       115073 non-null  str    
 4   order_approved_at              115059 non-null  str    
 5   order_delivered_carrier_date   114346 non-null  str    
 6   order_delivered_customer_date  113202 non-null  str    
 7   order_estimated_delivery_date  115073 non-null  str    
 8   payment_sequential             115073 non-null  int64  
 9   payment_type                   115073 non-null  str    
 10  payment_installments           115073 non-null  int64  
 11  payment_value                  115073 non-null  float64
 12  customer_unique_id             115073 non

## 4. 중복제거(.. 할까 말까?)
### 분석 목적이 주문단위라면.. 
`payment_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum'
}).reset_index()`
### 로 하는 것이 낫다고 함

In [14]:
olist_df[olist_df.duplicated(subset = 'order_id')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,...,product_category_name_english,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,seller_zip_code_prefix,seller_city,seller_state
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,3,voucher,...,housewares,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,9350,maua,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2,voucher,...,housewares,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,9350,maua,SP
11,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,1,credit_card,...,office_furniture,abc5655186d40772bd6e410420e6a3ed,5,NaN,NaN,2017-08-17 00:00:00,2017-08-18 01:47:32,8577,itaquaquecetuba,SP
13,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07 00:00:00,1,credit_card,...,garden_tools,15898b543726a832d4137fbef5d1d00e,1,NaN,Aguardando retorno da loja,2017-05-30 00:00:00,2017-05-30 23:13:47,29156,cariacica,ES
22,83018ec114eee8641c97e08f7b4e926f,7f8c8b9c2ae27bf3300f670c3d478be8,delivered,2017-10-26 15:54:26,2017-10-26 16:08:14,2017-10-26 21:46:53,2017-11-08 22:22:00,2017-11-23 00:00:00,3,voucher,...,telephony,219cf59cf889bc85babbd1cd1fe30f2d,4,NaN,NaN,2017-11-09 00:00:00,2017-11-10 01:06:29,12327,jacarei,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115061,9115830be804184b91f5c00f6f49f92d,da2124f134f5dfbce9d06f29bdb6c308,delivered,2017-10-04 19:57:37,2017-10-04 20:07:14,2017-10-05 16:52:52,2017-10-20 20:25:45,2017-11-07 00:00:00,1,credit_card,...,toys,ebd75732b5804e934123d11ec1f11db0,5,NaN,NaN,2017-10-21 00:00:00,2017-10-23 14:48:40,26020,nova iguacu,RJ
115062,9115830be804184b91f5c00f6f49f92d,da2124f134f5dfbce9d06f29bdb6c308,delivered,2017-10-04 19:57:37,2017-10-04 20:07:14,2017-10-05 16:52:52,2017-10-20 20:25:45,2017-11-07 00:00:00,2,voucher,...,toys,ebd75732b5804e934123d11ec1f11db0,5,NaN,NaN,2017-10-21 00:00:00,2017-10-23 14:48:40,26020,nova iguacu,RJ
115063,9115830be804184b91f5c00f6f49f92d,da2124f134f5dfbce9d06f29bdb6c308,delivered,2017-10-04 19:57:37,2017-10-04 20:07:14,2017-10-05 16:52:52,2017-10-20 20:25:45,2017-11-07 00:00:00,2,voucher,...,toys,ebd75732b5804e934123d11ec1f11db0,5,NaN,NaN,2017-10-21 00:00:00,2017-10-23 14:48:40,26020,nova iguacu,RJ
115065,aa04ef5214580b06b10e2a378300db44,f01a6bfcc730456317e4081fe0c9940e,delivered,2017-01-27 00:30:03,2017-01-27 01:05:25,2017-01-30 11:40:16,2017-02-07 13:15:25,2017-03-17 00:00:00,1,credit_card,...,health_beauty,e8995c053d3db2d9c07407efe7de52dd,5,NaN,NaN,2017-02-08 00:00:00,2017-02-11 12:37:36,80310,curitiba,PR


## 5. 날짜 컬럼 형변환후 연, 월, 요일별로 분해

In [16]:
# datetime 형식으로 구매 일자 변경
date_time = olist_df['order_purchase_timestamp'].str.split()

date_list = []
time_list = []
for x in range(date_time.shape[0]) :
    date_list.append(date_time[x][0])
    time_list.append(date_time[x][1])
    
olist_df['purchase_date'], olist_df['purchase_time'] = date_list, time_list
olist_df = olist_df.drop(columns = {'order_purchase_timestamp'})

olist_df['purchase_date'] = pd.to_datetime(olist_df['purchase_date'])

# 구매 일자를 연, 월, 요일별로 분해
olist_df['year'] = olist_df['purchase_date'].dt.year
olist_df['month'] = olist_df['purchase_date'].dt.month
olist_df['day_of_week'] = olist_df['purchase_date'].dt.day_name()

# olist_df['purchase_time']에 시간 값만 저장
olist_df['purchase_time'] = olist_df['purchase_time'].str.slice(0, 2)

## 6. 브라질의 주 이름의 한글 명을 컬럼으로 따로 저장(시각화에 용이하기 위해)

In [17]:
# 주 이름을 한글로 바꾸는 함수
def get_kor_state(state) :
    if state == 'AC' :
        state_kor_name = '아크리주'
    elif state == 'AL' :
        state_kor_name = '알라고아스주'
    elif state == 'AP' :
        state_kor_name = '아마파주'
    elif state == 'AM' :
        state_kor_name = '아마조나스주'
    elif state == 'BA' :
        state_kor_name = '바이아주'
    elif state == 'CE' :
        state_kor_name = '세아라주'
    elif state == 'DF' :
        state_kor_name = '연방구'
    elif state == 'ES' :
        state_kor_name = '이스피리투산투주'
    elif state == 'GO' :
        state_kor_name = '고이아스주'
    elif state == 'MA' :
        state_kor_name = '마라냥주'
    elif state == 'MT' :
        state_kor_name = '마투그로수주'
    elif state == 'MG' :
        state_kor_name = '미나스제라이스주'
    elif state == 'PA' :
        state_kor_name = '파라주'
    elif state == 'PB' :
        state_kor_name = '파라이바주'
    elif state == 'PR' :
        state_kor_name = '파라나주'
    elif state == 'PE' :
        state_kor_name = '페르남부쿠주'
    elif state == 'PI' :
        state_kor_name = '피아우이주'
    elif state == 'RJ' :
        state_kor_name = '리우데자네이루주'
    elif state == 'RN' :
        state_kor_name = '히우그란지두노르치주'
    elif state == 'RS' :
        state_kor_name = '히우그란지두술주'
    elif state == 'RO' :
        state_kor_name = '혼도니아주'
    elif state == 'RR' :
        state_kor_name = '호라이마주'
    elif state == 'SC' :
        state_kor_name = '산타카타리나주'
    elif state == 'SP' :
        state_kor_name = '상파울루주'
    elif state == 'SE' :
        state_kor_name = '세르지피주'
    elif state == 'MS' :
        state_kor_name = '마투그로수두술'
    else :
        state_kor_name = '토칸칭스주'
    return state_kor_name
    
# 주의 한글 이름 컬럼 추가
for index, row in olist_df.iterrows():
    state = row['customer_state']
    kor_name = get_kor_state(state)
    olist_df.at[index,'kor_state'] = kor_name

## 7. 주의 위치에 따라 지역별로 나누어 새로운 컬럼에 저장

In [18]:
# state를 입력받으면 지역을 return하는 함수
def get_region(state) :
    if (state == 'SP' or state == 'MG' or state == 'ES' or state == 'RJ') : 
        region = '남동부'
    elif (state == 'PR' or state == 'SC' or state == 'RS') : 
        region = '남부'
    elif (state == 'BA' or state == 'PE' or state == 'CE' or state == 'RN' or state == 'PI' or state == 'MA' or state == 'SE' or state == 'AL' or state == 'PB') : 
        region = '북동부'
    elif (state == 'GO' or state == 'MT' or state == 'MS' or state == 'DF') : 
        region = '중서부'
    else : 
        region = '북부'
    return region
    
# 판매자 거주 지역 컬럼 추가
for index, row in olist_df.iterrows():
    state = row['seller_state']
    region = get_region(state)
    olist_df.at[index,'seller_region'] = region

# 구매자 거주 지역 컬럼 추가
for index, row in olist_df.iterrows():
    state = row['customer_state']
    region = get_region(state)
    olist_df.at[index,'customer_region'] = region